# 07. Transferencia de aprendizaje con RoBERTuito

Se ajusta el modelo `pysentimiento/robertuito-sentiment-analysis` con el corpus peruano SMPS. RoBERTuito pertenece a la familia **RoBERTa/BERT** y fue preentrenado con texto de redes sociales en español. La muestra boliviana de 100 publicaciones no se usa en este cuaderno.

Referencias: [RoBERTuito](https://aclanthology.org/2022.lrec-1.785/), [pysentimiento](https://arxiv.org/abs/2106.09462) y [BERT](https://aclanthology.org/N19-1423/).


## Fundamentos que deben poder explicarse

### ¿Qué es inteligencia artificial?

Es el campo que diseña sistemas capaces de realizar tareas asociadas con capacidades humanas, como reconocer patrones o clasificar lenguaje. En esta tesis la tarea concreta es asignar una polaridad a un texto.

### ¿Qué es un modelo matemático?

Es una función con parámetros que transforma una entrada en una salida. Aquí puede resumirse como:

\[
f_{\theta}(texto) = (P(negativa), P(neutral), P(positiva))
\]

`θ` representa todos los valores aprendidos por la red.

### Arquitectura BERT/RoBERTa/RoBERTuito

1. El **tokenizador** divide el texto y produce identificadores.
2. Los **embeddings** convierten esos identificadores y sus posiciones en vectores.
3. Doce capas Transformer producen representaciones contextuales mediante autoatención y redes intermedias.
4. Una capa de clasificación genera tres *logits* o puntajes.
5. `softmax` convierte los logits en valores que suman 1; se selecciona la clase mayor.

La autoatención se expresa como:

\[
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

RoBERTuito no es BERT clásico: usa la estrategia RoBERTa y fue preentrenado para lenguaje de redes sociales en español.

### Capas intermedias y funciones de activación

Cada bloque Transformer contiene atención y una red neuronal intermedia. RoBERTuito utiliza **GELU** dentro de esas capas. En la salida se aplica **softmax**:

\[
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
\]

GELU ayuda a transformar representaciones internas; softmax sirve para comparar las tres clases. No cumplen la misma función.

### Parámetros e hiperparámetros

- **Parámetros:** valores que el entrenamiento aprende, como embeddings, matrices de atención y pesos del clasificador.
- **Hiperparámetros:** decisiones fijadas antes del entrenamiento, como tasa de aprendizaje, épocas, tamaño de lote y longitud máxima.

### Función de pérdida y función de costo

Para una publicación se usa entropía cruzada:

\[
L=-\sum_i y_i\log(p_i)=-\log(p_{clase\ correcta})
\]

La **pérdida** corresponde a un ejemplo; el **costo** es el promedio de pérdidas del lote o conjunto:

\[
J(\theta)=\frac{1}{m}\sum_{k=1}^{m}L_k
\]

### Gradiente y actualización

El gradiente `∇θJ(θ)` indica cómo cambia el costo al cambiar cada parámetro. La idea básica es:

\[
\theta_{t+1}=\theta_t-\eta\nabla_{\theta}J(\theta_t)
\]

donde `η` es la tasa de aprendizaje. El entrenamiento real usa **AdamW**, que adapta esta actualización usando el historial de gradientes y regularización por decaimiento de pesos.


## A. Instalar y cargar herramientas

- **Qué hace:** instala bibliotecas, fija semillas y exige una GPU CUDA antes de entrenar.
- **Para qué:** ejecutar el ajuste y poder repetir la partición y la inicialización.
- **Por qué:** sin versiones y semillas registradas, dos entrenamientos pueden producir resultados diferentes sin explicación.

**Configuración local:** el entorno `tesis-nlp313` debe mostrar `PyTorch 2.10.0+cu128` y `NVIDIA GeForce RTX 4060`. Si CUDA no está disponible, el notebook se detiene antes de gastar horas en CPU.


In [ ]:
%pip install -q --no-warn-script-location "transformers==4.49.0" "pysentimiento==0.7.3" "emoji==2.16.0" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn matplotlib


In [ ]:
from pathlib import Path

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB")
else:
    RAIZ = Path.cwd()

if not (RAIZ / "TABLAS").is_dir():
    raise FileNotFoundError("Abre este notebook desde la carpeta COLAB o ajusta RAIZ.")
print("Carpeta de trabajo:", RAIZ)

import inspect
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from pysentimiento.preprocessing import preprocess_tweet
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Reinicia Jupyter, selecciona el kernel tesis-nlp313 "
        "y comprueba que PyTorch termine en +cu128. No se iniciará entrenamiento con CPU."
    )

DISPOSITIVO = torch.device("cuda:0")
NOMBRE_GPU = torch.cuda.get_device_name(0)
MEMORIA_GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
torch.set_float32_matmul_precision("high")
torch.cuda.empty_cache()

MODELO_BASE = "pysentimiento/robertuito-sentiment-analysis"
CORPUS = "pyupeu/social-media-peruvian-sentiment"
SALIDA = RAIZ / "outputs/modelos/robertuito_smps_v4"
SALIDA.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Dispositivo:", DISPOSITIVO)
print("GPU:", NOMBRE_GPU, f"({MEMORIA_GPU_GB:.1f} GB)")


## B. Cargar SMPS

- **Qué hace:** descarga el corpus peruano etiquetado y separa entrenamiento de validación.
- **Para qué:** ajustar parámetros con `train` y decidir el mejor punto con `validation`.
- **Por qué:** la validación debe estar separada del aprendizaje y la prueba boliviana debe permanecer intacta.


In [ ]:
corpus = load_dataset(CORPUS)
if "train" not in corpus:
    raise ValueError("SMPS no contiene una partición train.")

if "validation" in corpus:
    train_raw, val_raw = corpus["train"], corpus["validation"]
else:
    division = corpus["train"].train_test_split(test_size=0.15, seed=SEMILLA)
    train_raw, val_raw = division["train"], division["test"]

if not {"text", "label"}.issubset(train_raw.column_names):
    raise ValueError(f"Columnas inesperadas en SMPS: {train_raw.column_names}")

mapa_smps = {"negative": 0, "neutral": 1, "positive": 2}
if "label_name" not in train_raw.column_names:
    raise ValueError("SMPS no incluye label_name para verificar el significado de sus números.")
for parte in (train_raw, val_raw):
    if any(mapa_smps.get(str(fila["label_name"]).lower()) != int(fila["label"]) for fila in parte):
        raise ValueError("La correspondencia de etiquetas de SMPS no es 0=negativa, 1=neutral, 2=positiva.")

print("Entrenamiento:", len(train_raw), "| Validación:", len(val_raw))


## C. Cargar la arquitectura y mostrar su configuración

- **Qué hace:** carga el tokenizador y los pesos iniciales, y muestra las capas reales del modelo.
- **Para qué:** vincular la explicación matemática con la configuración ejecutada.
- **Por qué:** no basta afirmar que se usó BERT; deben documentarse arquitectura, clases y dimensiones.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=3)

arquitectura = pd.DataFrame([
    ["familia", modelo.config.model_type],
    ["capas Transformer", modelo.config.num_hidden_layers],
    ["dimensión oculta", modelo.config.hidden_size],
    ["cabezas de atención", modelo.config.num_attention_heads],
    ["dimensión intermedia", modelo.config.intermediate_size],
    ["activación intermedia", modelo.config.hidden_act],
    ["tamaño del vocabulario", modelo.config.vocab_size],
    ["clases", modelo.config.num_labels],
], columns=["elemento", "valor"])
display(arquitectura)
print("Parámetros entrenables:", f"{sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}")


## D. Tokenizar

- **Qué hace:** convierte cada texto de SMPS en identificadores con el vocabulario de RoBERTuito.
- **Para qué:** producir las entradas numéricas de la red.
- **Por qué:** los embeddings solo pueden buscar vectores a partir de identificadores de su propio tokenizador.


In [ ]:
MAX_TOKENS = 128

def tokenizar(lote):
    textos = [preprocess_tweet(str(texto), lang="es") for texto in lote["text"]]
    salida = tokenizer(textos, truncation=True, max_length=MAX_TOKENS)
    salida["labels"] = lote["label"]
    return salida

train_ds = train_raw.map(tokenizar, batched=True, remove_columns=train_raw.column_names)
val_ds = val_raw.map(tokenizar, batched=True, remove_columns=val_raw.column_names)
print("Ejemplo tokenizado:", train_ds[0]["input_ids"][:20])


## E. Definir hiperparámetros y entrenar

- **Qué hace:** ajusta el modelo durante tres épocas con AdamW y selecciona el punto con mayor F1 macro de validación.
- **Para qué:** adaptar los parámetros ya aprendidos al corpus de sentimiento peruano.
- **Por qué:** F1 macro da el mismo peso a positiva, negativa y neutral, aunque las clases tengan tamaños distintos.

Las iteraciones reales dependen de `ceil(n_entrenamiento / 16) × 3`; no se confunden épocas con pasos.


In [ ]:
HIPERPARAMETROS = {
    "epocas": 3,
    "tasa_aprendizaje": 2e-5,
    "lote_entrenamiento": 16,
    "lote_evaluacion": 32,
    "max_tokens": MAX_TOKENS,
    "weight_decay": 0.01,
    "semilla": SEMILLA,
}
display(pd.DataFrame(HIPERPARAMETROS.items(), columns=["hiperparametro", "valor"]))

pasos_por_epoca = (len(train_ds) + HIPERPARAMETROS["lote_entrenamiento"] - 1) // HIPERPARAMETROS["lote_entrenamiento"]
pasos_totales = pasos_por_epoca * HIPERPARAMETROS["epocas"]
print("GPU de entrenamiento:", NOMBRE_GPU)
print("Pasos por época:", pasos_por_epoca, "| pasos totales:", pasos_totales)

def medir(prediccion):
    reales = prediccion.label_ids
    estimadas = np.asarray(prediccion.predictions).argmax(axis=1)
    return {
        "accuracy": accuracy_score(reales, estimadas),
        "f1_macro": f1_score(reales, estimadas, average="macro", zero_division=0),
    }

nombre_eval = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
argumentos = TrainingArguments(
    output_dir=str(SALIDA / "checkpoints"),
    num_train_epochs=HIPERPARAMETROS["epocas"],
    learning_rate=HIPERPARAMETROS["tasa_aprendizaje"],
    per_device_train_batch_size=HIPERPARAMETROS["lote_entrenamiento"],
    per_device_eval_batch_size=HIPERPARAMETROS["lote_evaluacion"],
    weight_decay=HIPERPARAMETROS["weight_decay"],
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    dataloader_pin_memory=True,
    report_to="none",
    seed=SEMILLA,
    **{nombre_eval: "epoch"},
)
nombre_tokenizer = "processing_class" if "processing_class" in inspect.signature(Trainer).parameters else "tokenizer"
entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=medir,
    **{nombre_tokenizer: tokenizer},
)
print("Dispositivo confirmado por Trainer:", entrenador.args.device)
resultado = entrenador.train()
print("Entrenamiento terminado:", resultado.metrics)


## F. Revisar los resultados del entrenamiento

- **Qué hace:** muestra los valores numéricos y las curvas obtenidas durante el entrenamiento.
- **Para qué:** comprobar cuánto disminuye la pérdida y cuánto alcanzan las métricas de validación.
- **Por qué:** la tabla permite leer cifras exactas y los gráficos permiten observar su evolución.

El gráfico se construye únicamente con el historial real producido por `Trainer`; no inventa valores.


In [ ]:
import matplotlib.pyplot as plt

historial = pd.DataFrame(entrenador.state.log_history)
entrenamiento = historial.dropna(subset=["loss"]) if "loss" in historial else pd.DataFrame()
validacion = historial.dropna(subset=["eval_f1_macro"]) if "eval_f1_macro" in historial else pd.DataFrame()

if not entrenamiento.empty:
    print(f"Pérdida inicial: {entrenamiento['loss'].iloc[0]:.4f}")
    print(f"Pérdida final:   {entrenamiento['loss'].iloc[-1]:.4f}")

if validacion.empty:
    print("No hay métricas de validación. Primero ejecuta completamente la celda de entrenamiento.")
else:
    nombres = {"epoch": "Época", "eval_loss": "Pérdida de validación", "eval_accuracy": "Accuracy", "eval_f1_macro": "F1 macro"}
    columnas = [columna for columna in nombres if columna in validacion.columns]
    tabla_metricas = validacion[columnas].rename(columns=nombres).round(4)
    print("Resultados numéricos por época:")
    display(tabla_metricas)
    mejor = validacion.loc[validacion["eval_f1_macro"].idxmax()]
    print(f"Mejor F1 macro: {mejor['eval_f1_macro']:.4f} (época {mejor['epoch']:.0f})")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
if not entrenamiento.empty:
    ejes[0].plot(entrenamiento["step"], entrenamiento["loss"], color="#2A6FBB", linewidth=2)
    ejes[0].set_title("Pérdida durante el entrenamiento")
    ejes[0].set_xlabel("Paso de actualización")
    ejes[0].set_ylabel("Entropía cruzada")
else:
    ejes[0].text(0.5, 0.5, "Sin registros de pérdida", ha="center", va="center")
    ejes[0].set_axis_off()

if not validacion.empty:
    eje_x = validacion["epoch"] if "epoch" in validacion else validacion["step"]
    ejes[1].plot(eje_x, validacion["eval_f1_macro"], marker="o", markersize=8, color="#2E8B57", linewidth=2)
    ejes[1].set_title("F1 macro de validación")
    ejes[1].set_xlabel("Época")
    ejes[1].set_ylabel("F1 macro")
    ejes[1].set_ylim(0, 1)
    for x, y in zip(eje_x, validacion["eval_f1_macro"]):
        ejes[1].annotate(f"{y:.4f}", (x, y), xytext=(0, 8), textcoords="offset points", ha="center")
else:
    ejes[1].text(0.5, 0.5, "Sin registros de validación", ha="center", va="center")
    ejes[1].set_axis_off()

for eje in ejes:
    eje.grid(alpha=0.25)
    eje.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
CARPETA_GRAFICOS = RAIZ / "TABLAS/07_entrenamiento"
CARPETA_GRAFICOS.mkdir(parents=True, exist_ok=True)
fig.savefig(CARPETA_GRAFICOS / "curvas_entrenamiento_v4.png", dpi=160, bbox_inches="tight")
plt.show()


## G. Guardar el modelo y la evidencia

- **Qué hace:** guarda los pesos finales y las métricas producidas por esta ejecución.
- **Para qué:** permitir que el cuaderno 08 evalúe exactamente el modelo entrenado.
- **Por qué:** una afirmación de entrenamiento necesita pesos, configuración y resultados verificables.


In [ ]:
MODELO_FINAL = SALIDA / "modelo_final"
entrenador.save_model(str(MODELO_FINAL))
tokenizer.save_pretrained(str(MODELO_FINAL))

evidencia = {
    "version": "v4",
    "modelo_inicial": MODELO_BASE,
    "corpus": CORPUS,
    "n_entrenamiento": int(len(train_raw)),
    "n_validacion": int(len(val_raw)),
    "hiperparametros": HIPERPARAMETROS,
    "metricas_entrenamiento": resultado.metrics,
    "metricas_validacion": entrenador.evaluate(),
    "modelo_final": str(MODELO_FINAL),
    "grafico": "TABLAS/07_entrenamiento/curvas_entrenamiento_v4.png",
}
CARPETA = RAIZ / "TABLAS/07_entrenamiento"
CARPETA.mkdir(parents=True, exist_ok=True)
(CARPETA / "resumen_entrenamiento_v4.json").write_text(
    json.dumps(evidencia, ensure_ascii=False, indent=2, default=float), encoding="utf-8"
)
print("Modelo guardado:", MODELO_FINAL)
